# Final Task B 17-class - EfficientNet-B3

Train EfficientNet-B3 directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = Path.cwd()

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "final-B"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"final-B: {FINAL_B_DIR}")


In [ ]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    !pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [ ]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "efficientnet_b3"
DISPLAY_NAME = "EfficientNet-B3"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


In [ ]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


In [ ]:
summarize_results(results)
